# ⚡ Combined Backtest — Classifier Entry + Regression TP/SL

Notebook ini menggabungkan **dua kekuatan model** dalam satu mesin backtest:

| Komponen | Peran | Sumber |
|----------|-------|--------|
| **Model Klasifikasi** (Buy the Dip) | Menentukan **KAPAN** masuk (Entry Timing) | `model_is_buy_dip_*.pkl` |
| **Model Regresi** (5 Label) | Menentukan **BAGAIMANA** masuk dan keluar (TP/SL/Sizing/Time-Stop) | `model_{target}_*.pkl` |
| **Regime Detector** (EMA 10/20/50) | Filter makro — blokir beli saat Downtrend, agresif saat Uptrend | Rule-Based |

**Alur Keputusan Harian:**
1. Classifier bilang "Beli"? → Lanjut ke langkah 2
2. Regime Detector izinkan? → Lanjut ke langkah 3
3. Model Regresi hitung TP/SL/RR → Risk Manager setuju? → **EKSEKUSI**

**Prasyarat:** Jalankan `main.ipynb` (model regresi) dan `modelling_classifier.ipynb` terlebih dahulu.

In [1]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_stock_data
from src.features import create_features
from src.label import create_labels
from src.model_config import FEATURES
from src.backtest import CombinedBacktester
from src.visualizer import plot_interactive_candlestick

# ==================== KONFIGURASI ====================
TICKER = 'BUMI.JK'
IN_SAMPLE_END = '2023-12-31'
OOS_START = '2024-01-01'
OOS_END = '2025-12-31'
TODAY = datetime.now().strftime('%Y-%m-%d')

# Target Regresi
REGRESSION_TARGETS = ['trend_slope', 'return', 'risk', 'days_to_max', 'days_to_min']
# Target Klasifikasi
CLASSIFIER_TARGET = 'is_buy_dip'

# Parameter Backtest
INITIAL_CAPITAL = 5_000_000
MAX_RISK_PCT = 0.06       # Menyesuaikan dengan setting regresi Anda (6%)
MAX_ALLOC_PCT = 0.50      # 50% modal per posisi
MIN_RR_RATIO = 1.1        # Minimal Risk-Reward Ratio
MIN_BUY_PROBA = 0.20      # Ambang batas probabilitas classifier

print(f"Ticker: {TICKER}")
print(f"Modal: Rp{INITIAL_CAPITAL:,.0f}")
print(f"Min RR Ratio: {MIN_RR_RATIO} | Min Buy Proba: {MIN_BUY_PROBA}")

Ticker: BUMI.JK
Modal: Rp5,000,000
Min RR Ratio: 1.1 | Min Buy Proba: 0.2


In [2]:
# ==================== LOAD & PREPARE DATA ====================
df_raw = load_stock_data(ticker=TICKER, start_date='2014-11-01', end_date=TODAY)
df = create_features(df_raw.copy())
df = create_labels(df)

# Split In-Sample dan Out-of-Sample
df_in_sample = df.loc[:IN_SAMPLE_END].copy()
df_oos = df.loc[OOS_START:OOS_END].copy()

# Fitur (X)
X_in = df_in_sample[FEATURES].copy()
X_oos = df_oos[FEATURES].copy()

# Target Regresi (y_reg) — Dictionary berisi 5 Series
y_in_reg_dict = {target: df_in_sample[target].copy() for target in REGRESSION_TARGETS}
y_oos_reg_dict = {target: df_oos[target].copy() for target in REGRESSION_TARGETS}

# Target Klasifikasi (y_cls) — 1 Series
y_in_cls = df_in_sample[CLASSIFIER_TARGET].copy()
y_oos_cls = df_oos[CLASSIFIER_TARGET].copy()

print(f"In-Sample  : {len(X_in)} baris")
print(f"Out-of-Sample: {len(X_oos)} baris")
print(f"\nLabel Positif (In-Sample) : {y_in_cls.sum():.0f} ({y_in_cls.mean()*100:.1f}%)")
print(f"Label Positif (OOS)       : {y_oos_cls.sum():.0f} ({y_oos_cls.mean()*100:.1f}%)")

Mengunduh data terbaru untuk BUMI.JK dari Yahoo Finance...
Berhasil memuat 2901 baris data untuk BUMI.JK.
Info Data Cleaning: Menghapus 123 baris data (NaN atau Volume 0).
Berhasil memuat 2778 baris data bersih untuk BUMI.JK.
In-Sample  : 2159 baris
Out-of-Sample: 473 baris

Label Positif (In-Sample) : 193 (8.9%)
Label Positif (OOS)       : 64 (13.5%)


In [3]:
# ==================== JALANKAN COMBINED BACKTEST ====================
backtester = CombinedBacktester(
    ticker=TICKER,
    initial_capital=INITIAL_CAPITAL,
    max_risk_pct=MAX_RISK_PCT,
    max_alloc_pct=MAX_ALLOC_PCT,
    min_rr_ratio=MIN_RR_RATIO,
    min_buy_proba=MIN_BUY_PROBA
)

df_trades, df_equity = backtester.run_backtest(
    X_in=X_in,
    y_in_reg_dict=y_in_reg_dict,
    y_in_cls=y_in_cls,
    X_oos=X_oos,
    y_oos_reg_dict=y_oos_reg_dict,
    y_oos_cls=y_oos_cls,
    df_raw_prices=df_raw,
    feature_cols=FEATURES
)

 SIMULASI COMBINED BACKTEST (BUMI.JK)
 Classifier Entry + Regression TP/SL/Sizing + Regime Detector
 Modal Awal: Rp5,000,000.00
[INIT] Model Classifier dimuat: ..\models\bumi\model_is_buy_dip_bumi.pkl
[INIT] Memuat 5 model regresi dari folder models/bumi...
--------------------------------------------------
PERIODE: 2024-01 | OOS: 22 Hari Bursa
--------------------------------------------------
[RETRAIN] Bulan pertama: menggunakan model awal tanpa retrain.
--------------------------------------------------
PERIODE: 2024-02 | OOS: 18 Hari Bursa
--------------------------------------------------
[RETRAIN] Melatih ulang 5 model regresi dengan 2181 baris data...
[RETRAIN] 5 model regresi berhasil diperbarui.
[RETRAIN] Melatih ulang model classifier dengan 2181 baris data...
[RETRAIN] Model classifier berhasil diperbarui.

--------------------------------------------------
PERIODE: 2024-03 | OOS: 18 Hari Bursa
--------------------------------------------------
[RETRAIN] Melatih ulang 5 mode

In [4]:
# ==================== DETAIL TRANSAKSI ====================
if df_trades is not None:
    print(f"\nTotal Transaksi: {len(df_trades)}")
    print(f"\n--- Exit Reason Distribution ---")
    print(df_trades['exit_reason'].value_counts())
    print(f"\n--- Ringkasan Profit per Trade ---")
    print(df_trades[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'lots', 'net_profit', 'roi_pct', 'exit_reason', 'days_held']].to_string())
else:
    print("Tidak ada transaksi yang dieksekusi.")


Total Transaksi: 10

--- Exit Reason Distribution ---
exit_reason
Take Profit       5
Time-Stop         4
Akhir Backtest    1
Name: count, dtype: int64

--- Ringkasan Profit per Trade ---
  entry_date  exit_date  entry_price  exit_price  lots  net_profit    roi_pct     exit_reason  days_held
0 2024-04-30 2024-05-17        101.0        92.0   247  -231723.05  -9.274702       Time-Stop         10
1 2024-09-04 2024-09-19        101.0       111.0   235   224918.50   9.462045     Take Profit         10
2 2024-09-25 2024-10-01        122.0       133.0   204   213883.80   8.580981     Take Profit          4
3 2024-10-07 2024-10-21        138.0       138.0   188   -10377.60  -0.399401       Time-Stop         10
4 2025-05-06 2025-05-15        110.0       119.0   235   200631.25   7.749739     Take Profit          5
5 2025-05-20 2025-06-05        125.0       123.0   215   -53642.50  -1.993010       Time-Stop         10
6 2025-10-03 2025-10-17        164.0       128.0   162  -592369.20 -22.26294

In [5]:
# ==================== VISUALISASI CHART INTERAKTIF ====================
plot_interactive_candlestick(
    df=df_raw,
    ticker_name=TICKER,
    start_date=OOS_START,
    trades_df=df_trades,
    equity_curve_df=df_equity,
    classifier_predictions_df=backtester.predictions_df
)

In [ ]:
# ==================== SIMPAN JURNAL TRADING ====================
if df_equity is not None and len(df_equity) > 0:
    final_equity = df_equity.iloc[-1]['total_equity']
    save_csv_path = f"../data/processed/jurnal_combined_{final_equity:,.2f}_{MAX_RISK_PCT}_{MAX_ALLOC_PCT}_{MIN_RR_RATIO}.csv"
    df_journal = backtester.generate_journal(df_raw, save_csv_path)
    display(df_journal)